# 08 - Persistencia en Firebase / Firestore

Este notebook documenta y prepara la carga de metadata del dataset, metricas de modelos, resultados comparativos y configuraciones experimentales en Firestore. No reentrena modelos y no modifica archivos en `data/raw/`.

## Por que Firestore

Firestore es una base NoSQL orientada a documentos. Es adecuada para esta etapa porque los resultados del proyecto tienen estructura semiestructurada: metadata del dataset, metricas por experimento, configuraciones de entrenamiento y resultados de validacion cruzada. Estos documentos pueden evolucionar sin exigir un esquema relacional rigido.

Persistir metricas y configuracion es clave para la trazabilidad del experimento. No alcanza con saber cual fue el mejor modelo: tambien se necesita guardar con que dataset, features, target, particion, pesos de clase y criterio de seleccion se obtuvo ese resultado.

## Colecciones creadas

- `datasets`: metadata del dataset procesado `siniestros_limpio_enriquecido`.
- `model_results`: un documento por experimento/modelo con metricas principales.
- `model_config`: un documento por configuracion experimental con features, preprocessing, split y notas metodologicas.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.upload_results_firebase import (
    DATA_FILE,
    LOG_FILE,
    build_firestore_payloads,
    upload_payloads,
)
from src.firebase_client import get_pipeline_logger, load_env_file

pd.set_option("display.max_columns", 100)

## Revision de metadata local

Antes de subir a Firestore se lee el CSV procesado para obtener `shape`, columnas y tipos de datos. Esta metadata permite saber exactamente que version analitica del dataset acompana a los resultados de modelado.

In [3]:
df_metadata = pd.read_csv(DATA_FILE, nrows=5)
full_shape = pd.read_csv(DATA_FILE).shape

print(f"Dataset procesado: {DATA_FILE.resolve()}")
print(f"Shape: {full_shape}")
display(df_metadata.dtypes.rename("dtype").to_frame())

Dataset procesado: C:\Users\Germán\Desktop\TP AVANZADA\tp-final-siniestros-viales\data\processed\siniestros_limpio_enriquecido.csv
Shape: (62076, 14)


,dtype
fecha_siniestro,object
anio_siniestro,int64
modo_desplazamiento_victima,object
sexo_victima,object
edad_victima,int64
gravedad_victima,object
rol_victima,float64
edad_grupo,object
es_mortal,int64
es_grave_o_mortal,int64


## Construccion de documentos

Los documentos se construyen a partir de los JSON ya generados en `outputs/`: `model_metrics.json`, `model_comparison.json` y `cross_validation_results.json`. Esta etapa solo persiste resultados existentes; no reentrena modelos.

In [4]:
payloads = build_firestore_payloads()

print("Documentos datasets:")
display(pd.DataFrame.from_dict(payloads["datasets"], orient="index"))

print("Documentos model_results:")
display(pd.DataFrame.from_dict(payloads["model_results"], orient="index"))

print("Documentos cross_validation:")
display(pd.DataFrame.from_dict(payloads["cross_validation"], orient="index"))

print("Documentos model_config:")
display(pd.DataFrame.from_dict(payloads["model_config"], orient="index"))

print("Eventos de logs:")
display(pd.DataFrame.from_dict(payloads["logs"], orient="index"))


Documentos datasets:


,nombre,fecha_carga,fecha_ejecucion,cantidad_filas,cantidad_columnas,columnas,dtypes,nulos_por_columna,fuente,ruta_local_archivo_procesado,dataset_hash_sha256,version_dataset,muestra_tipo,muestra_cantidad_filas,muestra_registros
siniestros_limpio_enriquecido,siniestros_limpio_enriquecido,2026-06-05T21:45:25.579962+00:00,2026-06-05T21:45:25.579962+00:00,62076,14,"[fecha_siniestro, anio_siniestro, modo_desplaz...","{'fecha_siniestro': 'object', 'anio_siniestro'...","{'fecha_siniestro': 0, 'anio_siniestro': 0, 'm...",data\processed\siniestros_limpio_enriquecido.csv,C:\Users\Germán\Desktop\TP AVANZADA\tp-final-s...,316a4d53ec7af9c9fe369771128b4476feb8800e13488e...,v1_enriquecido,primeras_200_filas,200,"[{'fecha_siniestro': '2019-01-01', 'anio_sinie..."


Documentos model_results:


,experiment_id,model_name,modelo_seleccionado,target,accuracy,precision,recall,f1,f1_mean,f1_std,cv_folds,selected_model,features,fecha_ejecucion,created_at,baseline,comparison_table,cross_validation,selection_criteria,selection_metric,metrics,excluded_features,preprocessing,class_weight,random_state,notes
modelo_ganador,modelo_ganador,RandomForestClassifier,RandomForestClassifier,es_grave_o_mortal,0.720361,0.139576,0.897727,0.24159,0.235999,0.001846,5.0,True,"{'numeric': ['anio_siniestro', 'mes_siniestro'...",2026-06-05T21:45:25.579962+00:00,2026-06-05T21:45:25.579962+00:00,"{'model_name': 'LogisticRegression', 'metrics'...","[{'modelo': 'RandomForestClassifier', 'accurac...",{'best_model_metrics': {'accuracy_mean': 0.719...,"[max_f1_mean, min_f1_std]",NaN,NaN,NaN,NaN,NaN,NaN,NaN
modelo_seleccionado_produccion,modelo_seleccionado_produccion,RandomForestClassifier,NaN,es_grave_o_mortal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,"{'numeric': ['anio_siniestro', 'mes_siniestro'...",NaN,2026-06-05T21:45:25.579962+00:00,NaN,NaN,NaN,NaN,f1_mean,"{'accuracy': 0.720360824742268, 'precision': 0...","[GRAVEdad_victima, gravedad_victima, es_mortal...",{'categorical': 'OneHotEncoder(handle_unknown=...,balanced,42.0,Modelo seleccionado por mayor F1 promedio y me...


Documentos cross_validation:


,target,hypothesis,cv_strategy,selection_criteria,best_model,best_model_metrics,features,data,comparison_table,fold_scores,dataset_id,created_at
resultados_validacion_cruzada,es_grave_o_mortal,Las caracteristicas de la victima y del contex...,"{'name': 'StratifiedKFold', 'n_splits': 5, 'sh...","[max_f1_mean, min_f1_std]",RandomForestClassifier,"{'accuracy_mean': 0.7192153265919593, 'accurac...","{'numeric': ['anio_siniestro', 'mes_siniestro'...","{'n_rows': 62076, 'n_features': 9, 'positive_r...","[{'modelo': 'RandomForestClassifier', 'accurac...",{'LogisticRegression': {'accuracy': [0.6368395...,siniestros_limpio_enriquecido,2026-06-05T21:45:25.579962+00:00


Documentos model_config:


,experiment_id,dataset_id,dataset_version,dataset_hash_sha256,target,modelo_seleccionado,modelos_evaluados,baseline_model,model_parameters,cv_strategy,features,features_usadas,features_numericas,features_categoricas,features_excluidas,preprocessing_config,fecha_ejecucion,created_at
configuracion_modelo_ganador,configuracion_modelo_ganador,siniestros_limpio_enriquecido,v1_enriquecido,316a4d53ec7af9c9fe369771128b4476feb8800e13488e...,es_grave_o_mortal,RandomForestClassifier,"[RandomForestClassifier, DecisionTreeClassifie...",LogisticRegression,"{'class_weight': 'balanced', 'random_state': 4...","{'name': 'StratifiedKFold', 'n_splits': 5, 'sh...","{'numeric': ['anio_siniestro', 'mes_siniestro'...","[anio_siniestro, mes_siniestro, dia_semana_sin...","[anio_siniestro, mes_siniestro, dia_semana_sin...","[modo_desplazamiento_victima, sexo_victima, ro...","[GRAVEdad_victima, gravedad_victima, es_mortal...",{'categorical_encoder': 'OneHotEncoder(handle_...,2026-06-05T21:45:25.579962+00:00,2026-06-05T21:45:25.579962+00:00


Eventos de logs:


,event,status,message,created_at,dataset_id,model_results_doc,cross_validation_doc,model_config_doc,predictions_doc
inicio_carga_firestore,inicio_carga_firestore,INFO,Inicio de persistencia de resultados del TP fi...,2026-06-05T21:45:25.579962+00:00,NaN,NaN,NaN,NaN,NaN
fin_carga_firestore,fin_carga_firestore,INFO,Persistencia completada correctamente.,2026-06-05T21:45:25.579962+00:00,siniestros_limpio_enriquecido,modelo_ganador,resultados_validacion_cruzada,configuracion_modelo_ganador,predicciones_experimento_actual


## Carga a Firestore

La carga real requiere credenciales locales. No deben subirse al repo. Configure un archivo `.env` ignorado por Git con `FIREBASE_CREDENTIALS_PATH` o use `GOOGLE_APPLICATION_CREDENTIALS` como variable de entorno.

Por seguridad, `RUN_UPLOAD` queda en `False`. Para ejecutar la escritura desde notebook, cambiarlo a `True` en un entorno local con credenciales configuradas.

In [5]:
RUN_UPLOAD = False

logger = get_pipeline_logger(LOG_FILE)
logger.info("Inicio de carga a Firebase/Firestore desde notebook")
load_env_file(PROJECT_ROOT / ".env")

if RUN_UPLOAD:
    upload_payloads(payloads, logger)
    logger.info("Carga a Firebase/Firestore desde notebook finalizada correctamente")
else:
    logger.info("Notebook ejecutado en modo revision; no se escribio en Firestore")
    print("Modo revision: no se escribio en Firestore.")

INFO | Inicio de carga a Firebase/Firestore desde notebook
INFO | Notebook ejecutado en modo revision; no se escribio en Firestore


Modo revision: no se escribio en Firestore.


## Ejecucion recomendada por script

Para produccion local se recomienda ejecutar el script versionado:

```bash
python scripts/upload_results_firebase.py --dry-run
python scripts/upload_results_firebase.py
```

El primer comando valida archivos y payloads sin escribir. El segundo crea o actualiza los documentos en Firestore.